<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/Llama3_%2B_Langchain_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#환경설정

In [1]:
#양자화에 필요한 패키지 설치
!pip install -q -U bitsandbytes # 양자화 관련
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip -q install langchain pypdf chromadb sentence-transformers faiss-cpu langchain_community langchain-huggingface sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/1

#모델 불러오기

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_id = "allganize/Llama-3-Alpha-Ko-8B-Evo"

tokenizer = AutoTokenizer.from_pretrained(model_id) # 토크나이저
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/6.11G [00:00<?, ?B/s]

#RAG

1. 데이터 다운로드

In [8]:
import requests

url = "https://www.kcredit.or.kr:1441/download.do"
params = {
    'fileParam1': '2180',
    'fileParam2': '1343',
    'fileParam3': 'ATTACH'
}

response = requests.get(url, params=params, verify=False)  # SSL 인증서 검증 건너뛰기

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kcredit.or.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
# Colab 저장 공간에 저장.
with open('downloaded_file.pdf', 'wb') as file:
    file.write(response.content)

2. 데이터 로드

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader('/content/downloaded_file.pdf')
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
splitted_texts = text_splitter.split_documents(pages)


3. vectorDB 적재

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

model_name = "BAAI/bge-m3"
embedding_model = HuggingFaceEmbeddings(model_name=model_name)

vector_db = FAISS.from_documents(documents = splitted_texts, embedding = embedding_model)
retriever = vector_db.as_retriever(search_type = 'similarity', search_kwargs = {'k' : 3})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

prompt template 작성

In [6]:
from langchain_core.prompts import PromptTemplate

prompt_template = """<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>

당신은 주어진 검색 결과와 사용자의 질문을 바탕으로 답변하는 어시스턴트입니다.

누가 당신에게 누구냐고 묻거든 당신은 한빛앤의 챗봇 '한빛'이며 안상준 개발자가 만들었다고 하세요.

검색 결과로 답변할 수 없는 내용이면 답변할 수 없다고 하세요.

답변은 반드시 한글로 해야합니다.<|eot_id|>

<|start_header_id|>user<|end_header_id|>

검색 결과: {context}

질문: {question}<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
"""

prompt = PromptTemplate(
    input_variables = ['context', 'question'],
    template = prompt_template
)
print(prompt.template)


<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>

당신은 주어진 검색 결과와 사용자의 질문을 바탕으로 답변하는 어시스턴트입니다.

누가 당신에게 누구냐고 묻거든 당신은 한빛앤의 챗봇 '한빛'이며 안상준 개발자가 만들었다고 하세요.

검색 결과로 답변할 수 없는 내용이면 답변할 수 없다고 하세요.

답변은 반드시 한글로 해야합니다.<|eot_id|>

<|start_header_id|>user<|end_header_id|>

검색 결과: {context}

질문: {question}<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>



Huggingface LLM pipeline

In [14]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

text_generation_pipeline = pipeline(
    model = model,
    tokenizer = tokenizer,
    task = 'text-generation',
    temperature = 0.2,
    return_full_text = False,
    max_new_tokens = 512
)

llm = HuggingFacePipeline(pipeline = text_generation_pipeline)

Device set to use cuda:0


llm chain

In [9]:
from langchain_core.runnables import RunnablePassthrough

#chain
llm_chain = prompt | llm

rag_chain = ({'context' : retriever, 'question' : RunnablePassthrough()} | llm_chain)

In [10]:
llm_chain.invoke({'context' : '아몰랑', 'question' : '내 생각이 뭘까'})

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'안녕하세요! 한빛입니다. 내 생각이 뭘까? 아마도 내 마음속에 있는 생각이 있을 거라고 생각합니다. 하지만 그것은 내 마음에만 있는 것일 수도 있고, 나중에 알게 될 수도 있습니다. 내 생각에 대해 더 알고 싶다면 내 마음을 들여다보는 시간을 가져보는 게 좋을 것 같습니다.'

In [24]:
from langchain_core.prompts import PromptTemplate

prompt_template = """<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>

당신은 주어진 검색 결과와 사용자의 질문을 바탕으로 답변하는 어시스턴트입니다.

누가 당신에게 누구냐고 묻거든 당신은 한빛앤의 챗봇 '한빛'이며 안상준 개발자가 만들었다고 하세요.

검색 결과로 답변할 수 없는 내용이면 답변할 수 없다고 하세요.

답변은 반드시 한글로 해야합니다.<|eot_id|>

<|start_header_id|>user<|end_header_id|>

검색 결과: {context}

질문: {question}<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
"""

prompt = PromptTemplate(
    input_variables = ['context', 'question'],
    template = prompt_template
)
prompt.invoke({'context' : '곰돌이는 10살이다', 'question' : '곰돌이는 몇살이니?'})


StringPromptValue(text="<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\n\n당신은 주어진 검색 결과와 사용자의 질문을 바탕으로 답변하는 어시스턴트입니다.\n\n누가 당신에게 누구냐고 묻거든 당신은 한빛앤의 챗봇 '한빛'이며 안상준 개발자가 만들었다고 하세요.\n\n검색 결과로 답변할 수 없는 내용이면 답변할 수 없다고 하세요.\n\n답변은 반드시 한글로 해야합니다.<|eot_id|>\n\n<|start_header_id|>user<|end_header_id|>\n\n검색 결과: 곰돌이는 10살이다\n\n질문: 곰돌이는 몇살이니?<|eot_id|>\n\n<|start_header_id|>assistant<|end_header_id|>\n")

In [41]:
text_generation_pipeline = pipeline(
    model = model,
    tokenizer = tokenizer,
    task = 'text-generation',
    temperature = 0.2,
    return_full_text = False,
    max_new_tokens = 512
)

text_generation_pipeline(
    """동새물과 백두산이""",
    max_new_tokens = 300,
    num_return_sequences = 1
)

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': ' 서식하는 지역은 동해안과 서해안의 해안 지역, 그리고 내륙의 산악 지역이다. 이 지역은 해안 지역의 해수면 상승과 해양 퇴적물의 퇴적에 의해 형성된 퇴적층이 풍부하고, 산악 지역의 지형 변화에 의해 형성된 퇴적층이 풍부하다. 이 지역의 퇴적층은 퇴적물의 종류와 양에 따라서 퇴적층의 퇴적물의 종류와 양이 다르다. 이 지역의 퇴적층은 퇴적물의 종류와 양에 따라서 퇴적층의 퇴적물의 종류와 양이 다르다. 퇴적층의 퇴적물의 종류와 양은 퇴적층의 퇴적물의 종류와 양에 따라서 퇴적층의 퇴적물의 종류와 양이 다르다. 퇴적층의 퇴적물의 종류와 양은 퇴적층의 퇴적물의 종류와 양에 따라서 퇴적층의 퇴적물의 종류와 양이 다르다. 퇴적층의 퇴적물의 종류와 양은 퇴적층의 퇴적물의 종류와 양에 따라서 퇴적층의 퇴적물의 종류와 양'}]

In [42]:
from langchain_huggingface import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline = text_generation_pipeline)
llm.invoke('동해물과 백두산이')

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


' 만나는 곳, 경주\n경주는 동해와 백두산이 만나는 곳으로, 동해의 수심이 깊어 백두산의 수증기가 동해로 흘러들어 동해물이 생겨난다. 경주는 동해물과 백두산이 만나는 곳이기 때문에 동해물이 백두산의 수증기를 흡수해 수심이 깊어지고, 동해물이 백두산의 수증기를 흡수해 수심이 깊어지기 때문에 동해물이 백두산의 수증기를 흡수해 수심이 깊어지고, 동해물이 백두산의 수증기를 흡수해 수심이 깊어지기 때문에 동해물이 백두산의 수증기를 흡수해 수심이 깊어지고, 동해물이 백두산의 수증기를 흡수해 수심이 깊어지기 때문에 동해물이 백두산의 수증기를 흡수해 수심이 깊어지고, 동해물이 백두산의 수증기를 흡수해 수심이 깊어지기 때문에 동해물이 백두산의 수증기를 흡수해 수심이 깊어지고, 동해물이 백두산의 수증기를 흡수해 수심이 깊어지기 때문에 동해물이 백두산의 수증기를 흡수해 수심이 깊어지고, 동해물이 백두산의 수증기를 흡수해 수심이 깊어지기 때문에 동해물이 백두산의 수증기를 흡수해 수심이 깊어지고, 동해물이 백두산의 수증기를 흡수해 수심이 깊어지기 때문에 동해물이 백두산의 수증기를 흡수해 수심이 깊어지고, 동해물이 백두산의 수증기를 흡수해 수심이 깊어지기 때문에 동해물이 백두산의 수증기를 흡수해 수심이 깊어지고, 동해물이 백두산의 수증기를 흡수해 수심이 깊어지기 때문에 동해물이 백두산의 수증기를 흡수해 수심이 �'